### Building a Question Answer Application Over a Graph Database

In [1]:
import os
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_groq import ChatGroq


load_dotenv()

neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME") 
neo4j_password = os.getenv("NEO4J_PASSWORD")
groq_api_key = os.getenv("GROQ_API_KEY")

c:\Users\viren\generative-ai-implementations\langchain\langchain-updated\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
graph = Neo4jGraph(url=neo4j_uri, username=neo4j_username, password=neo4j_password)
graph

In [3]:
movie_query = """ 
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' AS row
MERGE(m: Movie {movieId: toInteger(row.movieId)})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director IN split(row.director, '|') | 
    MERGE(p: Person {name: trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor IN split(row.actors, '|') | 
    MERGE(p: Person {name: trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre IN split(row.genres, '|') | 
    MERGE(g: Genre {name: trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))
"""

movie_query

" \nLOAD CSV WITH HEADERS FROM\n'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' AS row\nMERGE(m: Movie {movieId: toInteger(row.movieId)})\nSET m.released = date(row.released),\n    m.title = row.title,\n    m.imdbRating = toFloat(row.imdbRating)\nFOREACH (director IN split(row.director, '|') | \n    MERGE(p: Person {name: trim(director)})\n    MERGE (p)-[:DIRECTED]->(m))\nFOREACH (actor IN split(row.actors, '|') | \n    MERGE(p: Person {name: trim(actor)})\n    MERGE (p)-[:ACTED_IN]->(m))\nFOREACH (genre IN split(row.genres, '|') | \n    MERGE(g: Genre {name: trim(genre)})\n    MERGE (m)-[:IN_GENRE]->(g))\n"

In [4]:
graph.query(movie_query)

[]

In [5]:
graph.refresh_schema()
print(graph.schema)

Node properties:
Movie {title: STRING, released: DATE, movieId: INTEGER, imdbRating: FLOAT}
Person {name: STRING}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Movie)-[:IN_GENRE]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)


In [6]:
llm = ChatGroq(model="openai/gpt-oss-120b", api_key=groq_api_key)

In [7]:
chain = GraphCypherQAChain.from_llm(graph=graph, llm=llm, verbose=True, allow_dangerous_requests=True)
chain

GraphCypherQAChain(verbose=True, graph=<langchain_neo4j.graphs.neo4j_graph.Neo4jGraph object at 0x0000014A12971400>, cypher_generation_chain=PromptTemplate(input_variables=['examples', 'question', 'schema'], input_types={}, partial_variables={}, template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nExamples (optional):\n{examples}\n\nThe question is:\n{question}')
| _ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-

In [8]:
response = chain.invoke({"query": "Who was the director of the movie Casino"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: "Casino"})
RETURN p.name AS director
Full Context:
[{'director': 'Martin Scorsese'}]

> Finished chain.


{'query': 'Who was the director of the movie Casino',
 'result': 'Martin\u202fScorsese.'}

In [9]:
print(response['result'])

Martin Scorsese.


In [10]:
response = chain.invoke({"query": "Who were the actors of the movie Casino"})
print(response['result'])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:ACTED_IN]->(m:Movie {title: "Casino"}) RETURN p.name
Full Context:
[{'p.name': 'Robert De Niro'}, {'p.name': 'Joe Pesci'}, {'p.name': 'Sharon Stone'}, {'p.name': 'James Woods'}]

> Finished chain.
Robert De Niro, Joe Pesci, Sharon Stone, James Woods were the actors of the movie *Casino*.
